# Blind Sensors: Satellite PM2.5 Transfer

**Independent study.** Train where ground monitors exist; evaluate transfer into held-out cities (simulating sensor-blind regions) using meteorology + satellite-like AOD/proxy features.

This notebook builds a transparent multi-city table and measures in-city vs leave-city-out AUC/MAE.


In [1]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import GroupKFold
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
cities = ['Hanoi','HCM','Bangkok','Jakarta','Manila','Singapore','ChiangMai','CanTho']
rows=[]
for i,c in enumerate(cities):
    n=600
    aod = rng.uniform(0.05, 1.2, n)          # satellite-like aerosol optical depth
    blh = rng.uniform(200, 2000, n)          # boundary layer height
    wind = rng.uniform(0.5, 8, n)
    temp = rng.normal(28, 4, n)
    humid = rng.uniform(40, 95, n)
    # city bias = sensor network / emission regime
    city_bias = [-5, 8, 3, 12, -2, -8, 15, 6][i]
    pm = np.clip(18 + 55*aod - 0.008*blh - 1.2*wind + 0.3*(temp-25) + 0.05*humid + city_bias + rng.normal(0, 6, n), 1, None)
    rows.append(pd.DataFrame({'city':c,'aod':aod,'blh':blh,'wind':wind,'temp':temp,'humid':humid,'pm25':pm}))
df=pd.concat(rows, ignore_index=True)
df['unhealthy']=(df.pm25>=55.5).astype(int)
feats=['aod','blh','wind','temp','humid']

# In-city pooled CV (optimistic for deployment with local sensors)
from sklearn.model_selection import train_test_split
Xtr,Xte,ytr,yte = train_test_split(df[feats], df.pm25, test_size=0.2, random_state=0)
reg = HistGradientBoostingRegressor(max_depth=4, random_state=0).fit(Xtr,ytr)
pred=reg.predict(Xte)
print('In-distribution MAE', round(mean_absolute_error(yte,pred),2), 'R2', round(r2_score(yte,pred),3))

# Leave-city-out transfer (blind-sensor simulation)
mae_lco=[]; auc_lco=[]
gkf=GroupKFold(n_splits=len(cities))
for tr,te in gkf.split(df, df.pm25, groups=df.city):
    reg = HistGradientBoostingRegressor(max_depth=4, random_state=0).fit(df.iloc[tr][feats], df.iloc[tr].pm25)
    p = reg.predict(df.iloc[te][feats])
    mae_lco.append(mean_absolute_error(df.iloc[te].pm25, p))
    clf = HistGradientBoostingClassifier(max_depth=3, random_state=0).fit(df.iloc[tr][feats], df.iloc[tr].unhealthy)
    if df.iloc[te].unhealthy.nunique()>1:
        auc_lco.append(roc_auc_score(df.iloc[te].unhealthy, clf.predict_proba(df.iloc[te][feats])[:,1]))
print('Leave-city-out MAE mean', round(float(np.mean(mae_lco)),2), 'std', round(float(np.std(mae_lco)),2))
print('Leave-city-out unhealthy AUC mean', round(float(np.mean(auc_lco)),3))
pd.DataFrame({'city':cities,'mae':mae_lco}).assign(auc=pd.Series(auc_lco)).to_csv('blind_sensors_lco.csv', index=False)
print('\nIf LCO MAE >> in-distribution MAE, satellite features alone do not close the sensor gap.')


In-distribution MAE 8.13 R2 0.772


Leave-city-out MAE mean 8.9 std 2.93
Leave-city-out unhealthy AUC mean 0.975

If LCO MAE >> in-distribution MAE, satellite features alone do not close the sensor gap.


## Real data swap-in
- Ground truth: [OpenAQ](https://openaq.org/)
- Satellite AOD: MODIS MAIAC / Sentinel-5P
- Hold out entire metro areas with few/no monitors as the blind test set.
